# FALSETTO — Stage-1 training on a real GPU (Colab)

Trains the **Stage-1 segment detector** (frozen **MERT** features → **AudioCAT** head) and reports
real **Accuracy / Precision / Recall / F1 / AUC / Specificity** plus a **confusion matrix** on a
held-out test split.

**Runtime → Change runtime type → GPU (T4 is fine).**

Two data modes:

- **`proxy`** *(default, runs today, no downloads to request)* — real recorded music (an open HF
  dataset) vs. AI music generated on the fly with **MusicGen**. This is an *easier* task than the
  paper's benchmark, so treat the numbers as a **demonstration that the pipeline learns**, not as a
  reproduction of the paper.
- **`fakemusiccaps`** — the paper's actual Stage-1 benchmark. Needs a one-time
  [Zenodo access request](https://zenodo.org/records/13732524) for the fake clips (approved in ~a
  day) plus the MusicCaps real audio. Same notebook, just point `DATA_ROOT` at the folder.


## 1. Setup — clone, install, check GPU

In [ ]:
!nvidia-smi -L || echo "No GPU — set Runtime → Change runtime type → GPU"
!git clone -q https://github.com/arnavmahadev/Falsetto.git
%cd Falsetto
# Core install (MERT runs without nnAudio; it just skips the optional CQT branch).
!pip install -q . datasets
print("\nInstalled. If Colab warns about torch, it's fine — the preinstalled CUDA torch is kept.")

## 2. Choose the data mode + sizes

In [ ]:
DATA_MODE   = "proxy"          # "proxy" (runs now) or "fakemusiccaps"
N_PER_CLASS = 120              # clips per class; raise for stronger metrics (slower)
DATA_ROOT   = "data/raw/proxy" # for fakemusiccaps mode, point this at your real/ + model dirs
MANIFEST    = "data/manifests/fakemusiccaps_small.csv"
import os; os.makedirs("data/manifests", exist_ok=True)
print("mode:", DATA_MODE, "| clips/class:", N_PER_CLASS)

## 3. Build the dataset

### 3a. Real music (proxy mode) — stream an open HF music dataset
We pull a few hundred real recordings, trim each to 10 s, and save them under `real/`.


In [ ]:
if DATA_MODE == "proxy":
    import soundfile as sf, numpy as np
    from pathlib import Path
    from datasets import load_dataset

    real_dir = Path(DATA_ROOT) / "real"; real_dir.mkdir(parents=True, exist_ok=True)

    def stream_real(repo):
        ds = load_dataset(repo, split="train", streaming=True)
        for ex in ds:
            a = ex.get("audio")
            if a and "array" in a:
                yield np.asarray(a["array"], dtype="float32"), int(a["sampling_rate"])

    saved = 0
    for repo in ("lewtun/music_genres", "marsyas/gtzan"):   # primary, then fallback
        try:
            for y, sr in stream_real(repo):
                if y.ndim > 1: y = y.mean(axis=1)
                y = y[: sr * 10]                              # first 10 s
                if len(y) < sr * 3:  # skip too-short
                    continue
                sf.write(real_dir / f"real_{saved:04d}.wav", y, sr)
                saved += 1
                if saved >= N_PER_CLASS: break
            if saved >= N_PER_CLASS: break
        except Exception as e:
            print(f"  {repo} failed ({e}); trying next source...")
    print(f"saved {saved} real clips -> {real_dir}")

### 3b. AI music (proxy mode) — generate with MusicGen
Uses `transformers`' MusicGen (already installed). ~10 s per clip; a few hundred take a while, so
the default is modest. Files go under `musicgen/` so the manifest scanner labels them as AI.


In [ ]:
if DATA_MODE == "proxy":
    import torch, soundfile as sf
    from pathlib import Path
    from transformers import AutoProcessor, MusicgenForConditionalGeneration

    fake_dir = Path(DATA_ROOT) / "musicgen"; fake_dir.mkdir(parents=True, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    proc = AutoProcessor.from_pretrained("facebook/musicgen-small")
    mg = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(dev)
    sr = mg.config.audio_encoder.sampling_rate

    PROMPTS = [
        "an upbeat pop song with drums and synths", "a calm lo-fi hip hop beat",
        "an energetic rock track with electric guitar", "a smooth jazz piano piece",
        "an epic orchestral cinematic score", "a funky disco groove with bass",
        "a dreamy ambient electronic soundscape", "a latin salsa dance rhythm",
        "a country folk song with acoustic guitar", "a hard techno club track",
        "a soulful r&b ballad", "a reggae tune with offbeat guitar",
        "a classical string quartet", "an 80s synthwave retro track",
        "a bright indie pop melody",
    ]
    for i in range(N_PER_CLASS):
        prompt = PROMPTS[i % len(PROMPTS)]
        inputs = proc(text=[prompt], padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            audio = mg.generate(**inputs, do_sample=True, max_new_tokens=500)  # ~10 s
        wav = audio[0, 0].cpu().numpy()
        sf.write(fake_dir / f"gen_{i:04d}_musicgen.wav", wav, sr)
        if (i + 1) % 20 == 0: print(f"  generated {i+1}/{N_PER_CLASS}")
    print(f"saved {N_PER_CLASS} AI clips -> {fake_dir}")

> **FakeMusicCaps mode:** set `DATA_MODE="fakemusiccaps"` and `DATA_ROOT` to a folder laid out as
> `real/` + `MusicGen/` `MusicLDM/` `AudioLDM2/` `StableAudioOpen/` `Mustango/` (see
> `docs/DATASETS.md`). Skip 3a/3b and go straight to step 4.


## 4. Build a balanced, leak-free manifest (8:1:1 split)

In [ ]:
!python scripts/build_manifest.py --root "$DATA_ROOT" --out "$MANIFEST" \
    --max-per-class $N_PER_CLASS --seed 42

## 5. Train Stage-1 (frozen MERT → AudioCAT)
The extractor is frozen; only the AudioCAT head trains. First step downloads MERT (~350 MB, cached).


In [ ]:
!falsetto train --config configs/stage1_mert_fakemusiccaps_small.yaml --stage 1

## 6. Evaluate — metrics table + confusion matrix

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from falsetto.config import load_config
from falsetto.eval.table_stage1 import evaluate_checkpoint, results_to_markdown

cfg = load_config("configs/stage1_mert_fakemusiccaps_small.yaml")
ckpt = "checkpoints/stage1_mert_fakemusiccaps_small/best.pt"
res = evaluate_checkpoint(cfg, ckpt, split="test")

print(results_to_markdown({"MERT + AudioCAT": res}, title="Stage-1 (test split)"))

e = res.extra
cm = np.array([[e["tn"], e["fp"]], [e["fn"], e["tp"]]])
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["pred real", "pred AI"])
ax.set_yticks([0, 1], ["true real", "true AI"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=15)
ax.set_title(f"Stage-1 confusion (test) — acc={res.accuracy:.2f}  f1={res.f1:.2f}  auc={res.auc:.2f}")
fig.tight_layout()
import os; os.makedirs("results", exist_ok=True)
fig.savefig("results/stage1_confusion.png", dpi=150)
plt.show()
print("saved -> results/stage1_confusion.png  (download it for your README)")

## 7. Notes

- **Proxy numbers look high** because real-vs-MusicGen is easier than real-vs-many-generators. Report
  them as *"the Stage-1 pipeline trains and separates real from AI-generated audio"* — not as the
  paper's benchmark. For the faithful result, run mode `fakemusiccaps`.
- **Scale up** by raising `N_PER_CLASS` and the `epochs` in
  `configs/stage1_mert_fakemusiccaps_small.yaml`.
- **Save the checkpoint** (`checkpoints/stage1_mert_fakemusiccaps_small/best.pt`) to Drive if you want
  to reuse it — Colab wipes local files when the session ends.
